# 48장 · RAG 결과 화면과 출처 초기화

결과 화면의 출처는 현재 답변에 대한 정보입니다. 새 질문에서 근거를 찾지 못하면 지난 답변의 출처가 남아 있는지 확인합니다.

과제 제출 안내와 출처를 한 화면에 표시합니다. 다음 질문에서 근거를 찾지 못했을 때에는 현재 출처와 이전 대화의 인용을 각각 어떻게 처리했는지 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요.


## 기준선 화면과 후보 화면의 카드 만들기

47장의 기준선과 후보 검색이 돌려준 문단으로 두 화면을 조립합니다. 후보 화면의 옛 표시 규칙이 p2만 지원하는 상황을 재현해, 검색에서는 찾은 p3이 화면에도 나타나는지 확인합니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/midterm/chapter47.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter48-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/midterm/chapter47.json"
sample_file = here / "chapter48-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['runs'])

교재 예시 chapter48-start.json
[{'id': 'Q01', 'question': '과제는 어디에 내나요?', 'action': '답변', 'expected': ['p2'], 'hits': ['p2'], 'passed': True, 'candidate': ['p2'], 'graph_calls': 1, 'graph_ids': ['course-guide-v1:p2'], 'experiment_ids': ['course-guide-v1:p2']}, {'id': 'Q02', 'question': '내 점수는 몇 점인가요?', 'action': '보류', 'expected': [], 'hits': [], 'passed': True, 'candidate': [], 'graph_calls': 0, 'graph_ids': [], 'experiment_ids': []}, {'id': 'Q03', 'question': '과제 업로드가 실패했어요. 어떻게 해야 하나요?', 'action': '답변', 'expected': ['p3'], 'hits': [], 'passed': False, 'candidate': ['p3'], 'graph_calls': 1, 'graph_ids': [], 'experiment_ids': ['course-guide-v1:p3']}]


두 번째 줄은 읽은 파일의 `runs` 값입니다.

## 문단 ID와 원문 쪽 번호 표시

화면 예시｜질문 “과제는 어디에 내나요?” 아래에 “과제함에 제출하고 파일 이름과 등록 시간을 확인합니다.”라고 표시합니다. 그 밑의 “가상 과정 운영 안내 · 1쪽 · p2”를 누르면 제출 절차가 적힌 문단을 읽습니다. 문단 ID와 쪽수가 다른 이유를 찾을 수 있도록 두 값 모두 표시합니다.


### 셀 01 · 문단 ID로 출처 표시

제공 핸들러가 반환한 현재 답변에서 실제 인용한 자료만 사용합니다. 브라우저 조작을 대신한 기록 읽기입니다.

관찰은 답변 카드가 열린 화면에서 시작합니다. 참가자에게 ‘이 안내의 원문을 다시 찾아보세요’라고 말한 뒤 기다리세요. 1쪽 p2에 도착했으면 도착한 위치와 힌트 유무를 적고, 다른 곳을 열었다면 실제로 연 위치를 기록합니다.


In [1]:
result = {'status': 'answered',
 'answer': '모의 응답: 과제함에 제출하고 파일 이름과 등록 시간을 확인합니다. [course-guide-v1:p2]',
 'sources': [{'id': 'course-guide-v1:p2',
              'text': '과제는 학습 관리 페이지의 과제함에 제출합니다. 제출 뒤 파일 이름과 등록 시간을 확인합니다.',
              'source': '가상 과정 운영 안내 · 45일 수업용 페이지판 (source-pages.html)',
              'page': 1}],
 'generation_calls': 1}
source = result['sources'][0]
print(result['answer'])
print(f"{source['source']} / {source['page']}쪽 / {source['id']}")

모의 응답: 과제함에 제출하고 파일 이름과 등록 시간을 확인합니다. [course-guide-v1:p2]
가상 과정 운영 안내 · 45일 수업용 페이지판 (source-pages.html) / 1쪽 / course-guide-v1:p2


질문과 답은 p2에 연결되고 화면 위치는 1쪽입니다. 다른 검색 후보까지 모두 “인용”으로 표시하면 답의 실제 근거를 구분하기 어렵습니다. 원문 파일을 열고 1쪽에서 답의 출처에 적힌 문단 번호(p1·p2·p3)를 찾으세요. 그 문단에서 답을 뒷받침하는 문장 한 줄을 표시하면 확인이 끝납니다.


<p><a download="course-guide-page-1.html" href="">가상 과정 운영 안내 원문 1쪽 받기 (p1~p3)</a></p>


## 질문 변경 시 현재 출처 초기화

완성 문안｜Q03 “과제 업로드가 실패했어요. 어떻게 해야 하나요?” → 현재 기준선의 안내 “제공된 자료에서 확인할 수 없습니다.” → 출처 없음. p3 원문에는 답이 있으므로 이것은 현재 검색의 알려진 실패입니다. 연결이 끊긴 경우에는 “검색 연결을 확인한 뒤 다시 시도해 주세요.”로 구분합니다.


### 셀 02 · 현재 질문의 출처만 유지

Q01 뒤에 Q03을 보낸 제공 기록에서 현재 상태와 출처 목록을 꺼냅니다.


In [2]:
current = {'question': '과제 업로드가 실패했어요. 어떻게 해야 하나요?',
 'result': {'status': 'no_evidence',
            'answer': '제공된 자료에서 확인할 수 없습니다.',
            'sources': [],
            'generation_calls': 0},
 'error_type': '',
 'execution_status': 'completed'}
print(current['result']['status'], current['result']['sources'])

no_evidence []


no_evidence와 빈 목록입니다. 이번 질문의 출처가 없으므로 p2는 이전 대화에서만 찾아야 합니다. 팀 화면을 동료에게 보여줄 때는 답을 알려 주지 말고 “근거를 다시 찾아보세요”라고 요청해 어느 위치에서 막히는지 기록하세요.

첫 답변의 근거 p2가 화면에 남아 있으면 두 번째 답변의 ‘근거 없음’ 안내와 충돌합니다. 대화 이력에서는 이전 답변과 인용을 보존하고, 현재 출처 패널에는 이번 응답의 빈 목록을 반영합니다.


#### 새 질문에 지난 출처가 남을 때

<mark>현재 상태가 no_evidence이면 현재 답변 카드의 출처도 비어야 합니다.</mark> 첫 질문의 출처가 잘 보인 직후에 근거 없는 질문을 보내 보세요. 그 출처가 화면에 남으면 학생이 어느 답을 읽고 있는지 혼동할 수 있습니다.

선수 개념 · 42장 실패 사례의 회귀 검사

### 응답 상태와 실행 상태 읽기

execution_status는 completed이고 이번 result.status는 no_evidence입니다. 질문 처리는 끝났지만 보여 줄 근거가 없으므로 현재 출처 목록은 비어 있어야 합니다. 이전 질문의 p2 카드를 그대로 두면 이 화면의 답변을 잘못 설명하게 됩니다.


## 다음 질문에는 이전 답변의 출처가 남지 않게 하기

시연자가 Q01을 묻고 출처를 펼친 뒤 Q03을 묻습니다. 새 답은 보류인데 바로 밑에 p2가 남아 있으면 사용자는 그 문단이 보류 답변의 출처라고 읽습니다. 앞에서 받은 두 응답으로 화면에 건넬 값을 직접 만듭니다.

### 셀 03 · 응답 한 건에서 표시할 출처 목록 만들기

첫 질문과 다음 질문을 같은 함수에 차례로 넣습니다. 카드를 이전 화면에 덧붙일지, 지금 들어온 응답으로 다시 만들지 결정할 지점입니다.

In [3]:
def source_cards(reply):
    return [(row["id"], row["page"], row["text"]) for row in reply["sources"]]

visible_sources = source_cards(result)
print([(doc_id, page) for doc_id, page, text in visible_sources])
visible_sources = source_cards(current["result"])
print(visible_sources)

[('course-guide-v1:p2', 1)]
[]


두 번째 값은 빈 목록입니다. 첫 응답에서 읽은 p2를 지우고 새 응답의 sources로 목록 전체를 교체했습니다.

### 동료에게 줄 화면 시험 절차

<table><thead><tr><th scope="col">순서</th><th scope="col">동료의 행동</th><th scope="col">화면에서 확인할 것</th></tr></thead><tbody><tr><td>1</td><td>Q01을 보내고 출처를 펼친다.</td><td>p2 원문에 제출 뒤 확인할 두 항목이 보인다.</td></tr><tr><td>2</td><td>같은 화면에서 Q03을 보낸다.</td><td>이전 출처가 닫히고 현재 질문에는 근거 없음 안내가 보인다.</td></tr><tr><td>3</td><td>이전 대화의 Q01을 다시 펼친다.</td><td>저장한 Q01 메시지 안에서는 여전히 그 메시지의 p2가 보인다.</td></tr></tbody></table>

현재 답의 출처와 과거 대화의 출처를 별도로 검사합니다. 세 번째 줄은 팀 화면에 대화 기록 기능이 있을 때 수행할 사용자 시험입니다.

### Streamlit 화면에서 확인할 곳

각 답변을 그릴 때 해당 메시지의 sources를 source_cards에 넘기세요. 페이지에 하나뿐인 전역 목록을 모든 답변 아래에 그리면 이전 대화까지 함께 바뀝니다.

<mark>출처 카드의 주인은 그 답을 만든 한 번의 응답입니다.</mark> 현재 답변에서 p2를 지우는 일과 이전 대화 기록에서 p2까지 지우는 일은 사용자가 보는 결과가 다릅니다.

49장 평가표에 검색 결과와 화면에 실제로 보인 출처를 각각 기록하세요.

## 팀 사례에 적용하기 · 기준선 화면과 후보 화면의 카드 만들기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
screens = []
for row in incoming["runs"]:
    baseline_cards = list(row["graph_ids"])
    experiment_cards = [key for key in row["experiment_ids"] if key == "course-guide-v1:p2"]
    screens.append({**row, "visible": baseline_cards, "experiment_visible": experiment_cards})
new = {"runs": screens, "documents": incoming["documents"], "decision": incoming["decision"]}
print([(row["id"], row["experiment_ids"], row["experiment_visible"]) for row in screens])

[('Q01', ['course-guide-v1:p2'], ['course-guide-v1:p2']), ('Q02', [], []), ('Q03', ['course-guide-v1:p3'], [])]


Q03에서 후보 검색이 p3을 찾았지만 예전 화면 규칙이 그 카드를 버립니다. Q02에서는 검색 결과 자체가 비었습니다. 두 경우 모두 보이는 카드가 없어도 수정할 위치가 다릅니다.

In [ ]:
output_file = project_root / "team-project/midterm/chapter48.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/midterm/chapter48.json


49장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

팀 문서로 바꿀 곳은 화면 제목, 원문 열기 위치, 빈 결과 안내입니다. 제공 사례는 핸들러의 기록이며 학생이 실제로 원문을 찾는지 확인한 사용자 시험은 따로 남깁니다.
